# 🎬 Bollywood Movie Chatbot — Experimentation Notebook

This notebook covers:
1. Dataset EDA
2. NLP preprocessing experiments
3. Model training & evaluation
4. Recommendation engine testing
5. Visualisations

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['figure.facecolor'] = '#0d0d0d'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print('Setup complete ✅')

## 1. Dataset EDA

In [ ]:
from src.config import CSV_FILE

df = pd.read_csv(CSV_FILE)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print('\nNull counts:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)

In [ ]:
print('Descriptive statistics:')
df.describe()

### 1.1 Genre Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

genre_counts = df['genre'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(genre_counts)))
axes[0].bar(genre_counts.index, genre_counts.values, color=colors)
axes[0].set_title('Movies by Genre', fontsize=14, color='white')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].pie(genre_counts.values, labels=genre_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Genre Distribution (%)', fontsize=14, color='white')

plt.tight_layout()
plt.savefig('../outputs/eda_genre_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 IMDb Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['imdb'] = pd.to_numeric(df['imdb'], errors='coerce')

axes[0].hist(df['imdb'].dropna(), bins=20, color='#e94560', edgecolor='white', alpha=0.85)
axes[0].axvline(df['imdb'].mean(), color='#f5a623', linestyle='--', lw=2, label=f'Mean: {df["imdb"].mean():.1f}')
axes[0].set_title('IMDb Rating Distribution', fontsize=14)
axes[0].set_xlabel('IMDb Rating')
axes[0].set_ylabel('Frequency')
axes[0].legend()

avg_rating_by_genre = df.groupby('genre')['imdb'].mean().sort_values(ascending=False)
axes[1].barh(avg_rating_by_genre.index, avg_rating_by_genre.values,
             color='#3a7bd5', edgecolor='white')
axes[1].set_title('Average IMDb Rating by Genre', fontsize=14)
axes[1].set_xlabel('Average IMDb Rating')
for i, v in enumerate(avg_rating_by_genre.values):
    axes[1].text(v + 0.05, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/eda_imdb_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.3 Movies per Year

In [ ]:
df['year'] = pd.to_numeric(df['year'], errors='coerce')
year_counts = df['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(year_counts.index, year_counts.values, color='#e94560', edgecolor='none', alpha=0.8)
ax.set_title('Movies Released per Year', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Movies')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/eda_movies_per_year.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 Language Distribution

In [ ]:
lang_counts = df['language'].value_counts()
colors = plt.cm.tab10(np.linspace(0, 1, len(lang_counts)))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(lang_counts.index, lang_counts.values, color=colors, edgecolor='white')
ax.set_title('Movies by Language', fontsize=14)
ax.set_xlabel('Language')
ax.set_ylabel('Count')
for bar, v in zip(bars, lang_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            str(v), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/eda_language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. NLP Preprocessing Exploration

In [ ]:
from src.preprocessing import tokenize, lemmatize, clean_sentence, bag_of_words

sample = "Suggest some funny Hindi movies from 2020 with Shah Rukh Khan"
print('Input:', sample)
print('\nTokenized:', tokenize(sample))
print('\nCleaned + Lemmatized:', clean_sentence(sample))

In [ ]:
lemma_examples = [
    ('running', 'verb'), ('movies', 'noun'), ('suggested', 'verb'),
    ('romantic', 'adj'), ('watching', 'verb'), ('comedies', 'noun')
]
print('Lemmatisation examples:')
print(f'{"Word":<15} {"Lemma":<15}')
print('-' * 30)
for word, _ in lemma_examples:
    print(f'{word:<15} {lemmatize(word):<15}')

In [ ]:
from src.utils import load_intents, extract_entities
from src.preprocessing import build_vocabulary, encode_training_data

intents_data = load_intents()
words, classes, documents = build_vocabulary(intents_data)

print(f'Vocabulary size : {len(words)}')
print(f'Intent classes  : {len(classes)}')
print(f'Training samples: {len(documents)}')
print(f'\nClasses: {classes}')
print(f'\nSample vocab (first 20): {words[:20]}')

In [ ]:
X, y = encode_training_data(documents, words, classes)
print(f'X shape: {X.shape}  (samples × vocab_size)')
print(f'y shape: {y.shape}  (samples × n_classes)')
print(f'\nClass distribution:')
class_counts = y.sum(axis=0).astype(int)
for cls, cnt in sorted(zip(classes, class_counts), key=lambda x: -x[1]):
    bar = '█' * cnt
    print(f'  {cls:<25} {bar} ({cnt})')

### 2.1 BoW Sparsity Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# BoW heatmap (first 50 samples, first 80 vocab words)
axes[0].imshow(X[:50, :80], aspect='auto', cmap='Blues', interpolation='nearest')
axes[0].set_title('BoW Matrix (first 50 samples × 80 vocab words)', fontsize=12)
axes[0].set_xlabel('Vocabulary Index')
axes[0].set_ylabel('Sample Index')
plt.colorbar(axes[0].images[0], ax=axes[0])

# Class distribution bar chart
axes[1].barh(classes, class_counts, color='#e94560', edgecolor='white')
axes[1].set_title('Training Samples per Intent Class', fontsize=12)
axes[1].set_xlabel('Count')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('../outputs/nlp_bow_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Entity Extraction Testing

In [ ]:
test_queries = [
    "Suggest funny Hindi movies from 2020",
    "Best action movies with Akshay Kumar",
    "Romantic Bollywood films with high IMDb rating",
    "Show me Shah Rukh Khan movies from 2023",
    "Scary horror movies in Tamil",
    "Best biographical films",
    "What are good comedy movies",
]

print(f'{"Query":<50} {"Mood":<12} {"Genre":<12} {"Year":<6} {"Language":<10} {"Actor"}')
print('-' * 110)
for q in test_queries:
    e = extract_entities(q)
    print(f'{q:<50} {str(e["mood"]):<12} {str(e["genre"]):<12} {str(e["year"]):<6} {str(e["language"]):<10} {str(e["actor"])}')

## 4. Train Model

In [ ]:
from src.train import train
train()

### 4.1 Training Curve

In [ ]:
from PIL import Image
img = Image.open('../outputs/training_curve.png')
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

### 4.2 Confusion Matrix

In [ ]:
img = Image.open('../outputs/confusion_matrix.png')
plt.figure(figsize=(14, 12))
plt.imshow(img)
plt.axis('off')
plt.title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Chatbot Inference Testing

In [ ]:
from src.chatbot import MovieChatbot

bot = MovieChatbot()
print('Chatbot loaded ✅')

In [ ]:
test_messages = [
    "Hello",
    "Suggest some comedy movies",
    "I want romantic movies",
    "Best thriller films",
    "Show me movies with Shah Rukh Khan",
    "Funny Hindi movies from 2019",
    "Family friendly movies for kids",
    "Top rated Bollywood movies",
    "Movies similar to 3 Idiots",
    "Thanks, goodbye",
]

for msg in test_messages:
    result = bot.chat(msg)
    print(f'\n[User] {msg}')
    print(f'[Intent] {result["intent"]} ({result["confidence"]:.1%})')
    print(f'[Bot]  {result["response"]}')
    print(f'[Movies] {[m["title"] for m in result["movies"]]}')
    print(f'[Why]  {result["explanation"]}')

## 6. Recommendation Engine Exploration

In [ ]:
from src.recommendation_engine import RecommendationEngine

engine = RecommendationEngine()
print(f'Movies in database: {len(engine.df)}')
print(f'Genres available: {sorted(engine.df["genre"].unique())}')
print(f'Languages available: {sorted(engine.df["language"].unique())}')
print(f'Year range: {engine.df["year"].min()} – {engine.df["year"].max()}')
print(f'IMDb range: {engine.df["imdb"].min()} – {engine.df["imdb"].max()}')

In [ ]:
def show_movies(movies, title=''):
    if title:
        print(f'\n{'='*60}')
        print(f'  {title}')
        print(f'{'='*60}')
    for i, m in enumerate(movies, 1):
        actors = ', '.join(m.get('actors', [])[:2])
        print(f'{i:2}. {m["title"]:40} ⭐{m["imdb"]:4.1f}  {m["genre"]:12} {m["year"]}  {actors}')

show_movies(engine.recommend_by_genre('Comedy'), 'Top Comedy Movies')
show_movies(engine.recommend_by_mood('romantic'), 'Romantic Mood Movies')
show_movies(engine.recommend_by_rating(8.0), 'IMDb ≥ 8.0')
show_movies(engine.recommend_by_actor('Aamir Khan'), 'Aamir Khan Movies')

In [ ]:
show_movies(
    engine.recommend_combined(genre='Action', min_imdb=7.0, year_from=2015),
    'Action + IMDb≥7.0 + After 2015'
)

show_movies(
    engine.recommend_similar('3 Idiots'),
    'Movies similar to "3 Idiots" (TF-IDF Cosine Similarity)'
)

### 6.1 TF-IDF Similarity Heatmap

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

top_movies = engine.df.nlargest(15, 'imdb')
top_indices = top_movies.index.tolist()
top_titles  = top_movies['title'].tolist()

sub_matrix = engine._tfidf_matrix[top_indices]
sim_matrix = cosine_similarity(sub_matrix, sub_matrix)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    sim_matrix,
    xticklabels=top_titles,
    yticklabels=top_titles,
    cmap='Blues',
    annot=True,
    fmt='.2f',
    ax=ax,
    linewidths=0.5,
    cbar_kws={'label': 'Cosine Similarity'}
)
ax.set_title('TF-IDF Cosine Similarity — Top 15 Rated Movies', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/tfidf_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Evaluation Summary

In [ ]:
report_path = '../outputs/reports/classification_report.txt'
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())
else:
    print('Run Section 4 (Train Model) first to generate the report.')

## 8. Confidence Distribution Analysis

In [ ]:
import tensorflow as tf
from src.preprocessing import bag_of_words, load_artifacts
from src.config import MODEL_FILE, CONFIDENCE_THRESHOLD

model = tf.keras.models.load_model(MODEL_FILE)
vocab, intent_classes = load_artifacts()

eval_queries = [
    ("Hi",                            "greeting"),
    ("Recommend comedy movies",       "comedy_movies"),
    ("Thriller suspense films",       "thriller_movies"),
    ("Best action Bollywood",         "action_movies"),
    ("Romantic love story",           "romantic_movies"),
    ("I want to cry emotional movie", "emotional_movies"),
    ("Scary horror ghost film",       "horror_movies"),
    ("Family kids safe movie",        "family_movies"),
    ("Highest IMDb rated films",      "high_rated_movies"),
    ("Latest new release",            "latest_movies"),
    ("Old classic vintage cinema",    "old_classics"),
    ("Aamir Khan movies",             "actor_movies"),
    ("Goodbye see you later",         "goodbye"),
]

correct = 0
print(f'{"Query":<40} {"Expected":<20} {"Predicted":<20} {"Confidence":<10} {"OK?"}')
print('-' * 105)
for query, expected in eval_queries:
    bow = bag_of_words(query, vocab)
    preds = model.predict(bow[np.newaxis], verbose=0)[0]
    idx   = np.argmax(preds)
    predicted  = intent_classes[idx]
    confidence = preds[idx]
    ok = '✅' if predicted == expected else '❌'
    if predicted == expected:
        correct += 1
    print(f'{query:<40} {expected:<20} {predicted:<20} {confidence:<10.2%} {ok}')

print(f'\nAccuracy on eval set: {correct}/{len(eval_queries)} = {correct/len(eval_queries):.1%}')